# 19.11 网络效应实验 / Network Experiments (SUTVA & Cluster Randomization)

**中文**:Part 19 收官。19.1 说随机 A/B 是因果黄金标准,但它悄悄依赖一个假设——**SUTVA(个体处理值稳定假设)**:*"一个用户的处理,不影响另一个用户的结果。"* 在**社交网络、市场、双边平台**里,这个假设**被彻底打破**:如果给你开了"一键分享"功能(处理),你分享得多了,你的好友(哪怕在对照组)也会看到更多内容——**处理通过网络'溢出(spillover)'到了对照组**。此时标准 A/B 会**系统性地估错**效应。本节讲这个陷阱和它的解药——**聚类随机化**。
**English**: The finale of Part 19. 19.1 said randomized A/B is the causal gold standard, but it quietly relies on one assumption — **SUTVA (Stable Unit Treatment Value Assumption)**: *"one user's treatment doesn't affect another user's outcome."* In **social networks, marketplaces, and two-sided platforms**, this is **utterly broken**: if you get a "one-tap share" feature (treatment) and share more, your friends (even those in control) see more content — **the treatment "spills over" through the network into the control group**. Then standard A/B **systematically mis-estimates** the effect. This section covers the trap and its cure — **cluster randomization**.

---

**中文**:**SUTVA 被违反(网络干扰/溢出)时会发生什么？** 想象一个让用户多发内容的新功能:
**English**: **What happens when SUTVA is violated (network interference/spillover)?** Imagine a feature that makes users post more:
- **总效应** = 直接效应(你自己因功能而多发)+ 溢出效应(你的好友因为你多发而多看/多互动)。这才是"全量上线"真正的影响。
  **Total effect** = direct effect (you post more because of the feature) + spillover (your friends see/engage more because you post more). This is what "full rollout" truly does.
- **个体级 A/B**:随机给每个用户开/不开功能。问题:**对照组用户的好友里有一半是实验组**,所以对照组也吃到了溢出、结果被"抬高"了。于是"实验组 − 对照组"的差**把溢出效应抵消掉了**,只剩下直接效应——**严重低估总效应**(甚至可能把一个有效功能误判为无效)。
  **Individual-level A/B**: randomly enable/disable per user. Problem: **half of a control user's friends are in treatment**, so control also absorbs spillover and is "lifted." So "treatment − control" **cancels out the spillover**, leaving only the direct effect — **severely underestimating the total effect** (possibly misjudging an effective feature as useless).

**中文**:**解药:聚类随机化(cluster randomization)**。不按个人、而是按**社区/簇**(好友团、地理区域、供需市场)整体随机分配处理。这样一个用户的**大部分邻居和他在同一组**——实验社区内部的溢出被完整捕捉、对照社区几乎不受污染。于是估计能逼近真实总效应。**代价**:有效样本量从"用户数"骤降到"社区数",方差大增(所以要更多流量、更长时间)。
**English**: **The cure: cluster randomization.** Randomize treatment by **community/cluster** (friend groups, geographic regions, supply-demand markets) as a whole, not per individual. Then most of a user's **neighbors are in the same arm** — spillover within treated communities is fully captured, and control communities are barely contaminated. So the estimate approaches the true total effect. **The cost**: the effective sample size drops from "number of users" to "number of clusters," greatly increasing variance (needing more traffic and longer duration).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 社交/平台DS必考）**
> **中文**:**SUTVA=个体间无干扰**(标准 A/B 的隐含前提)。社交/市场/双边平台里被**网络溢出**打破→**个体级 A/B 低估总效应**(对照组吃到溢出, 差值抵消溢出只剩直接效应)。**解药=聚类随机化**:按社区/地理/市场整体分组, 捕捉簇内溢出。**代价**:有效样本=簇数(方差大增, 需更多流量)。相关设计:**ego-cluster(自我簇)、图聚类随机化、双边随机化、switchback(时间轮换, 用于市场/供需)**。判断要不要担心 SUTVA:问"一个用户的处理会不会影响别人的结果?"(社交功能、供需匹配、内容曝光 = 会)。溢出本身有时是**要估的目标**(病毒传播、网络外部性)。
> **English**: **SUTVA = no interference between units** (standard A/B's implicit premise). Broken by **network spillover** in social/market/two-sided platforms → **individual-level A/B underestimates the total effect** (control absorbs spillover; the difference cancels spillover, leaving only the direct effect). **Cure = cluster randomization**: assign treatment by community/geo/market as a whole to capture within-cluster spillover. **Cost**: effective sample = number of clusters (much higher variance, needs more traffic). Related designs: **ego-clusters, graph cluster randomization, two-sided randomization, switchback (time-alternating, for marketplaces)**. To decide if SUTVA matters, ask "does one user's treatment affect others' outcomes?" (social features, supply-demand matching, content exposure = yes). Spillover itself is sometimes **the quantity you want to estimate** (viral spread, network externalities).


In [ ]:

# ============================================================
# 模拟:带社区结构的社交图 + 溢出效应 / social graph with communities + spillover
# 中文:随机块模型生成"好友团"结构(社区内连接密, 社区间稀疏)。
#      结果 = 基线 + 直接效应*自己是否处理 + 溢出*处理邻居的比例。溢出很大。
# English: a stochastic block model gives "friend groups" (dense within, sparse across communities).
#      Outcome = baseline + direct*own treatment + spillover*(fraction of treated neighbors). Spillover is large.
# ============================================================
import numpy as np, networkx as nx, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
n_comm, per = 20, 25; N=n_comm*per                          # 20 个社区 × 25 人 / 20 communities × 25
probs=[[0.35 if i==j else 0.004 for j in range(n_comm)] for i in range(n_comm)]  # 块内密, 块间疏 / block probs
G=nx.stochastic_block_model([per]*n_comm, probs, seed=1)
comm=np.repeat(np.arange(n_comm), per)                       # 每个节点所属社区 / community of each node
A=nx.to_numpy_array(G); deg=A.sum(1); A_norm=A/np.maximum(deg[:,None],1)   # 行归一化邻接(算邻居比例)/ row-normalized
direct, spillover = 1.0, 2.0                                 # 直接效应 & 溢出效应 / direct & spillover
true_total = direct + spillover                             # 全量上线的真实总效应 / true total effect
def outcome(T):
    frac_treated_nb = A_norm @ T                            # 每人处理邻居的比例(向量化)/ fraction of treated neighbors
    return 5 + direct*T + spillover*frac_treated_nb + rng.normal(0,0.5,N)
print(f"社交图:{N} 用户, {n_comm} 社区, {G.number_of_edges()} 条边")
print(f"直接效应 {direct} + 溢出效应 {spillover} = 真实总效应 {true_total}")


**中文**:现在对比两种实验设计。**个体级随机化**:每个用户独立抛硬币。**聚类随机化**:整个社区一起抛硬币(同社区同处理)。各跑多次取平均,看谁能还原真实总效应 3.0。
**English**: Now compare two experiment designs. **Individual randomization**: each user flips a coin independently. **Cluster randomization**: each whole community flips one coin (same arm within a community). Run each many times and average to see which recovers the true total effect 3.0.


In [ ]:

# ============================================================
# 个体级 vs 聚类随机化 / individual vs cluster randomization
# ============================================================
def individual_ab():
    T=rng.integers(0,2,N).astype(float)                     # 每个用户独立随机 / per-user random
    Y=outcome(T); return Y[T==1].mean()-Y[T==0].mean()
def cluster_ab():
    comm_treat=rng.integers(0,2,n_comm)                     # 每个社区随机 / per-community random
    T=comm_treat[comm].astype(float)
    Y=outcome(T); return Y[T==1].mean()-Y[T==0].mean()

ind=[individual_ab() for _ in range(200)]; clu=[cluster_ab() for _ in range(200)]
print(f"真实总效应 / true total effect: {true_total:.2f}")
print(f"个体级 A/B:  均值 {np.mean(ind):.2f} ± {np.std(ind):.2f}  ← 严重低估(只抓到直接效应~{direct})!")
print(f"聚类随机化:  均值 {np.mean(clu):.2f} ± {np.std(clu):.2f}  ← 接近真值(捕捉了簇内溢出)")
print(f"\n为什么:个体级实验里, 对照组用户的邻居约一半是实验组 → 对照组也吃到溢出 → 差值把溢出抵消了")
print(f"注意:聚类随机化的【有效样本=社区数(20)而非用户数(500)】—— 一般会显著增大方差, 需更多流量/时间")


**中文**:个体级 A/B 只估到 ~1.0(直接效应),把 2.0 的溢出**完全漏掉**——如果按它决策,你会以为这个功能"效果一般"而毙掉它,实际上全量上线能带来 3 倍的效果。聚类随机化估到 ~2.4,接近真值。下面可视化两种设计下的处理分配,和它们的估计。
**English**: Individual A/B estimates only ~1.0 (the direct effect), **entirely missing** the 2.0 spillover — deciding by it, you'd think the feature is "mediocre" and kill it, when a full rollout would actually deliver 3× the effect. Cluster randomization estimates ~2.4, close to the truth. Below we visualize the treatment assignment under each design and their estimates.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,5))
pos=nx.spring_layout(G,seed=3,k=0.15)
# ① 个体级随机化:处理散落, 对照有很多处理邻居 / individual: scattered
T_ind=rng.integers(0,2,N)
nx.draw_networkx_nodes(G,pos,node_color=["#C44E52" if t else "#4C72B0" for t in T_ind],node_size=15,ax=ax[0])
nx.draw_networkx_edges(G,pos,alpha=0.05,ax=ax[0])
ax[0].set_title("个体级随机化:红蓝混杂→对照有处理邻居(溢出污染)"); ax[0].axis("off")
# ② 聚类随机化:整个社区同色 / cluster: whole communities same color
ct=rng.integers(0,2,n_comm); T_clu=ct[comm]
nx.draw_networkx_nodes(G,pos,node_color=["#C44E52" if t else "#4C72B0" for t in T_clu],node_size=15,ax=ax[1])
nx.draw_networkx_edges(G,pos,alpha=0.05,ax=ax[1])
ax[1].set_title("聚类随机化:整簇同组→簇内溢出被捕捉"); ax[1].axis("off")
# ③ 估计对比 / estimate comparison
ax[2].bar(["个体级\nindividual","聚类\ncluster","真值\ntruth"],[np.mean(ind),np.mean(clu),true_total],
          yerr=[np.std(ind),np.std(clu),0],capsize=6,color=["#C44E52","#55A868","#4C72B0"])
ax[2].axhline(true_total,ls="--",color="#4C72B0"); ax[2].axhline(direct,ls=":",color="gray")
ax[2].text(0,direct+0.05,"直接效应 only",fontsize=8,color="gray")
for i,v in enumerate([np.mean(ind),np.mean(clu),true_total]): ax[2].text(i,v,f"{v:.2f}",ha="center",va="bottom",fontsize=9)
ax[2].set_title("只有聚类随机化还原总效应 / only cluster recovers total"); ax[2].set_ylabel("估计的总效应")
plt.tight_layout(); plt.savefig("/tmp/ci11_viz.png",dpi=80); plt.show()
print("个体级(红蓝混杂)对照组吃溢出→低估; 聚类(整簇同色)捕捉簇内溢出→接近真值(但方差更大)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **网络溢出让"黄金标准"的 A/B 也会骗你**:个体级随机化明明是随机的、无混杂的,却把 3.0 的总效应估成了 1.0——**因为 SUTVA 被违反了**。对照组用户被实验组好友"传染",结果被抬高,一减,溢出就没了。这是社交产品做实验最隐蔽、最致命的坑:**一个能病毒式传播、全量上线效果拔群的功能,可能在个体级 A/B 里显得平平无奇而被砍掉。**
2. **聚类随机化是解药,但有代价**:按社区整体分组,让溢出留在组内被捕捉,估计逼近真值(~2.6 vs 真值 3.0——没到满分,因为社区间还有少量边让溢出泄漏)。**代价很实在**:①**有效样本量骤降**——从 500 个用户变成 **20 个社区**,一般会显著增大方差(要更多流量、更长时间才能达到同样功效;本例因为个体级估计本身被溢出搅得很不稳,方差对比不明显,但"有效样本=簇数"这个原则要牢记);②**如何切社区**是难题(社交图没有干净的边界),要用图聚类算法(如 Louvain, Part 16),且要平衡"簇要大到装下溢出"和"簇要多到有统计功效"。
3. **诚实的全景**:①**溢出方向不总是正的**——有的功能会造成"零和"竞争(你多了别人就少了,如信息流曝光),这时个体级 A/B 会**高估**;②**两边随机化(two-sided)** 用于双边市场(司机/乘客、买家/卖家);③**switchback(时间轮换)** 用于溢出走时间维度的场景(如动态定价、供需匹配——整个城市在处理/对照间轮换);④有时**溢出本身就是你要测的东西**(病毒系数、网络效应)。判断的第一步永远是问:*"一个用户的处理,会不会通过某种渠道影响别人的结果?"*——只要答案是"会",标准 A/B 就要打问号。

**English**:
1. **Network spillover fools even the "gold-standard" A/B**: individual randomization is genuinely random and unconfounded, yet it estimates the 3.0 total effect as 1.0 — **because SUTVA is violated**. Control users are "infected" by their treatment-arm friends, their outcomes lifted, and the subtraction erases the spillover. This is the most insidious and deadly trap for experimentation in social products: **a feature that spreads virally with a huge full-rollout impact can look mediocre in an individual-level A/B and get killed.**
2. **Cluster randomization is the cure, but it costs**: grouping by whole communities keeps spillover within arms to be captured, and the estimate approaches the truth (~2.6 vs 3.0 — not perfect, since cross-community edges still leak some spillover). **The cost is real**: ① **the effective sample plummets** — from 500 users to **20 communities**, generally raising variance a lot (needing more traffic and longer for the same power; in this particular sim the individual estimator is itself noisy from spillover so the variance contrast is muted, but remember the principle "effective sample = #clusters"); ② **how to cut communities** is hard (social graphs have no clean boundaries), requiring graph-clustering algorithms (e.g. Louvain, Part 16) and balancing "clusters large enough to contain spillover" vs "enough clusters for statistical power."
3. **The honest full picture**: ① **spillover isn't always positive** — some features create "zero-sum" competition (your gain is another's loss, e.g. feed impressions), where individual A/B **overestimates**; ② **two-sided randomization** for two-sided marketplaces (drivers/riders, buyers/sellers); ③ **switchback (time-alternating)** for spillover along the time dimension (dynamic pricing, supply-demand matching — a whole city alternates between treatment and control); ④ sometimes **the spillover is exactly what you want to measure** (viral coefficient, network effects). The first step is always to ask: *"can one user's treatment affect others' outcomes through some channel?"* — whenever the answer is "yes," standard A/B deserves scrutiny.

> 💼 **实战视角 / Practical angle**
> **中文**:网络实验是**社交/平台公司(Meta/LinkedIn/Uber/滴滴)** 的硬核难题。落地要点:①**先判断有没有干扰**(社交功能、供需、内容曝光通常有);②有干扰用**聚类随机化 / 图聚类分桶 / ego-cluster**;双边市场用**双边随机化或 switchback**;③用图聚类算法(Louvain)切簇, 权衡簇大小(装溢出) vs 簇数量(功效);④聚类实验**方差大**, 用聚类稳健标准误(按簇聚类), 且预留更多流量;⑤有时用**溢出的部分暴露**(只处理一个人的部分好友)来分离直接与溢出效应。面试金句:*"SUTVA 假设个体间无干扰, 社交网络里被溢出打破→个体级 A/B 低估总效应(对照吃溢出); 用聚类随机化按社区整体分组捕捉簇内溢出, 代价是有效样本=簇数、方差大增; 双边市场用双边随机化或 switchback。"*
> **English**: Network experiments are a hardcore challenge for **social/platform companies (Meta/LinkedIn/Uber/DiDi)**. Deployment keys: ① **first judge whether interference exists** (social features, supply-demand, content exposure usually do); ② with interference use **cluster randomization / graph-cluster bucketing / ego-clusters**; for two-sided marketplaces use **two-sided randomization or switchback**; ③ cut clusters with graph algorithms (Louvain), trading cluster size (contain spillover) vs count (power); ④ cluster experiments have **high variance** — use cluster-robust SEs (cluster by community) and reserve more traffic; ⑤ sometimes use **partial exposure of spillover** (treat only some of a person's friends) to separate direct from spillover effects. Interview line: *"SUTVA assumes no interference; social-network spillover breaks it → individual-level A/B underestimates the total effect (control absorbs spillover); cluster randomization groups by whole communities to capture within-cluster spillover, at the cost of effective sample = number of clusters and much higher variance; two-sided marketplaces use two-sided randomization or switchback."*

---
### 小结 / Summary
- **中文**:SUTVA=个体间无干扰; 社交溢出违反它→个体级 A/B 低估总效应(对照组吃溢出抵消)。
- **English**: SUTVA = no interference; social spillover violates it → individual-level A/B underestimates the total effect (control absorbs spillover).
- **中文**:聚类随机化按社区整体分组捕捉簇内溢出, 逼近真值; 代价=有效样本降为簇数、方差大增。
- **English**: Cluster randomization groups by whole communities to capture within-cluster spillover, approaching the truth; cost = effective sample drops to #clusters, variance rises.
- **中文**:双边市场用双边随机化/switchback; 先问"一个人的处理会不会影响别人的结果", 是就别用标准 A/B。
- **English**: Two-sided marketplaces use two-sided randomization/switchback; first ask "can one's treatment affect others' outcomes" — if yes, don't use standard A/B.

---
**中文**:🎉 至此 **Part 19 · 因果推断与实验** 全部完成!从 A/B 测试设计与分析 → 赌博机权衡 → 因果图(混杂/中介/对撞)→ 倾向匹配 → 双重差分 → 工具变量 → 回归断点 → 因果森林 → 提升建模 → 网络实验,你已经**从零实现**了因果推断的完整方法论,建立了"相关≠因果""随机化是黄金标准""不能随机时如何逼近实验"的核心思维——这是数据科学从"预测"走向"决策"的关键一跃。
**English**: 🎉 **Part 19 · Causal Inference & Experimentation** is complete! From A/B design and analysis → bandit trade-offs → causal DAGs (confounder/mediator/collider) → propensity matching → difference-in-differences → instrumental variables → regression discontinuity → causal forests → uplift modeling → network experiments, you have **implemented from scratch** the full methodology of causal inference, building the core mindset — correlation ≠ causation, randomization is the gold standard, how to approximate an experiment when you can't randomize — the crucial leap of data science from "prediction" to "decision."
